<div style="text-align: center;">
    <img src="https://mejores.com/wp-content/uploads/2020/08/Universidad-Tecnica-Federico-Santa-Maria.jpg" title="Title text" width="20%" height="20%" />
</div>



<hr style="height:2px;border:none"/>
<h1 align='center'> EIN092B Visualización-2024 </h1>

<H3 align='center'> Librería Mapas - Introducción a los datos geoespaciales en Python </H3>
<hr style="height:2px;border:none"/>

**Objetivos de aprendizaje:**
  * **Visualización de datos geoespaciales en python:** Personalizar gráficos geoespaciales en Python utilizando la biblioteca `MapClassify`, `folium` y `geopandas`.


<hr style="height:2px;border:none"/>

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import geopandas as gpd
#import geodatasets

## Importar datos geoespaciales

Los datos geoespaciales suelen estar disponibles en formatos de archivo SIG o almacenes de datos específicos, como shapefiles ESRI, archivos GeoJSON, archivos geopackage, base de datos PostGIS (PostgreSQL), entre otros...

Podemos utilizar la biblioteca GeoPandas para leer los formatos de archivo SIG, utilizando la función `geopandas.read_file`.

Por ejemplo, vamos a empezar leyendo un shapefile con todos los países del mundo (tomado de http://www.naturalearthdata.com/downloads/110m-cultural-vectors/110m-admin-0-countries/), e inspeccionar los datos.

Seleccionamos las siguientes columnas: 
- **ADM0_A3**: Código ISO Alpha-3 del país (tres letras).
- **SOVEREIGNT**: Nombre del país o estado soberano.
- **CONTINENT**: Continente donde está ubicado el país.
- **POP_EST**: Población estimada del país.
- **GDP_MD**: PIB estimado en millones de dólares.
- **geometry**: Geometría del país, usada para representar su forma en un mapa.


In [ ]:
# Carga el shapefile con GeoPandas
countries = gpd.read_file(r"C:\Users\jorge\Downloads\ne_110m_admin_0_countries.zip") # actualizar el path
# or if the archive is unpacked:
#countries = gpd.read_file(r"C:\Users\jorge\Downloads\ne_110m_admin_0_countries.shp") 

countries = countries[['ADM0_A3', 'SOVEREIGNT', 'CONTINENT', 'POP_EST', 'GDP_MD', 'geometry']]
print(countries.head())

In [ ]:
countries.crs

In [ ]:
countries.plot()

In [ ]:
countries.explore()

In [ ]:
!pip install folium

¿Qué observamos?:

- Usando `.head()`, podemos ver las primeras filas del conjunto de datos, al igual que lo hacemos con Pandas.
- Hay una columna `geometry` y los diferentes países están representados como polígonos.
- Podemos usar el método `.plot()` (matplotlib) o `explore()` (Folium / Leaflet.js) para obtener rápidamente una *visualización básica* de los datos.

## ¿Qué es un GeoDataFrame?

Utilizamos la biblioteca GeoPandas para leer los datos geoespaciales, y esto nos devolvió un `GeoDataFrame`:


In [ ]:
type(countries)

Un GeoDataFrame contiene un conjunto de datos tabulares y geoespaciales:

* Tiene una **columna 'geometry'** que contiene la información de geometría (o características en GeoJSON).
* Las otras columnas son los **atributos** (o propiedades en GeoJSON) que describen cada una de las geometrías.

Un `GeoDataFrame` es como un `DataFrame` de pandas, pero con algunas funcionalidades adicionales para trabajar con datos geoespaciales:

* Un atributo `.geometry` que siempre devuelve la columna con la información de geometría (devolviendo una GeoSeries). El nombre de la columna no necesariamente tiene que ser 'geometry', pero siempre será accesible a través del atributo `.geometry`.
* Tiene algunos métodos extra para trabajar con datos espaciales (área, distancia, buffer, intersección, ...), que aprenderemos en notebooks posteriores.

In [ ]:
countries.geometry

### Tipo de datos:

El atributo `.geometry` devuelve una **GeoSeries**, que es un tipo especial de **pandas Series** diseñado para trabajar con geometrías espaciales.  
Las geometrías pueden ser de diferentes tipos, como:

- **Point** (puntos).
- **LineString** (líneas).
- **Polygon** (polígonos).
- **MultiPolygon**, **MultiLineString**, **MultiPoint** (versiones múltiples de las anteriores).

<img src="https://mborne.github.io/cours-patron-conception/annexe/tp-geometry/schema/geometries-light.png" title="Title text" width="50%" height="50%" />


### Conversión de Sistemas de Coordenadas de Referencia (CRS) en GeoPandas

Cuando trabajamos con datos geoespaciales en GeoPandas, las geometrías de un `GeoDataFrame` tienen un **Sistema de Coordenadas de Referencia (CRS)** asociado, el cual define cómo las coordenadas en ese conjunto de datos se interpretan en un mapa.


El método **`to_crs()`** de GeoPandas se utiliza para **reproyectar** o convertir las geometrías de un `GeoDataFrame` de un CRS a otro. Esto es especialmente importante cuando:

- Necesitamos realizar **cálculos precisos de distancias o áreas**, ya que muchas proyecciones geográficas (como **EPSG:4326**) están basadas en grados de latitud y longitud, que no son adecuados para mediciones lineales.
- Queremos mostrar los datos en una proyección específica para una correcta representación en un mapa.


#### EPSG

En GeoPandas, EPSG se refiere a un código que identifica un sistema de referencia espacial (SRS) o sistema de coordenadas. EPSG significa European Petroleum Survey Group, que es la organización responsable de la creación y mantenimiento de una base de datos de códigos estándar para sistemas de referencia.

Los códigos EPSG son ampliamente utilizados en los sistemas de información geográfica (SIG) para definir cómo se proyectan y representan los datos espaciales en la Tierra. Un código EPSG es un número que corresponde a un sistema específico de coordenadas.


- **EPSG:4326** es uno de los más comunes y se refiere al sistema de coordenadas **WGS84** (utilizado en GPS), que utiliza latitudes y longitudes.  
- **EPSG:3857** es otro código común que se refiere a la proyección **Web Mercator**, utilizada en servicios de mapas web como Google Maps y OpenStreetMap.


#### ¿Qué es **EPSG:3857**?

**EPSG:3857** es el código EPSG que representa la proyección **Web Mercator**, una de las proyecciones más usadas en la visualización de mapas en línea (como en Google Maps y OpenStreetMap). La unidad de esta proyección es el **metro**, lo que la hace útil para calcular áreas y distancias.

#### Ejemplo: `countries.to_crs(epsg=3857)`

```python
# Convertir las geometrías a la proyección EPSG:3857 (Web Mercator)
countries_3857 = countries.to_crs(epsg=3857)
```

### Manipulación de geometrías:

Una vez que accedes al atributo `.geometry`, puedes realizar operaciones geoespaciales como:

- **Cálculo del área** (`gdf.geometry.area`).
- **Distancias entre geometrías** (`gdf.geometry.distance(other_geometry)`).
- **Intersecciones**, **uniones**, y otras operaciones geométricas.


In [ ]:
countries.crs

In [ ]:
# Reproyectar a EPSG:3857 (metros)

countries = countries.to_crs(epsg=3857)
print(countries.crs)
areas = countries.geometry.area

areas.head()

In [ ]:
countries.crs

In [ ]:
countries.head(1)

In [ ]:
countries['area'] = countries.geometry.area
countries['area_km2'] = countries['area'] / 1e6
print(countries[['SOVEREIGNT', 'area_km2']].head())

In [ ]:
countries.explore("area", legend = 'False')

**Sigue siendo un DataFrame**, por lo que tenemos toda la funcionalidad de Pandas disponible para utilizar en el conjunto de datos geoespaciales, y para hacer manipulaciones de datos con los atributos y la información geométrica juntos.

Por ejemplo, podemos calcular el número medio de población en todos los países (accediendo a la columna 'pop_est', y llamando al método `mean` sobre ella):

In [ ]:
mean_pop_est = countries['POP_EST'].mean()
max_pop_est = countries['POP_EST'].max()
min_pop_est = countries['POP_EST'].min()

# Imprimir los resultados
print(f"La población media estimada es: {mean_pop_est}")
print(f"La población máxima estimada es: {max_pop_est}")
print(f"La población mínima estimada es: {min_pop_est}")

También podemos utilizar el filtrado booleano para seleccionar un subconjunto del marco de datos en función de una condición:

In [ ]:
sur_america = countries[countries['CONTINENT'] == 'South America']
sur_america.plot(color = 'teal');

In [ ]:
sur_america = countries[countries['CONTINENT'] == 'South America']

fig, ax = plt.subplots(figsize=(10, 10))


sur_america.plot(column='POP_EST', cmap='bwr', legend=True, 
                 legend_kwds={'label': "Población Estimada",
                              'orientation': "horizontal"}, ax=ax)

ax.set_title('Mapa de Sudamérica con Población Estimada', fontsize=15, fontweight='bold')

for x, y, label in zip(sur_america.geometry.centroid.x, sur_america.geometry.centroid.y, sur_america['SOVEREIGNT']):
    ax.text(x, y, label, fontsize=8, ha='center', color='darkblue', fontweight='bold')

Importemos otros conjuntos de datos con distintos tipos de objetos geométricos.

Un conjunto de datos sobre ciudades del mundo (tomado de http://www.naturalearthdata.com/downloads/110m-cultural-vectors/110m-populated-places/), compuesto por datos de puntos:

In [ ]:
cities = gpd.read_file(r"C:\Users\jorge\Downloads\ne_110m_populated_places_simple.zip")
#cities = geopandas.read_file("data/ne_110m_populated_places.zip")
cities = cities[['name',  'geometry']]
cities.head()

In [ ]:
cities.explore()

Y un conjunto de datos de los ríos del mundo (disponible en [http://www.naturalearthdata.com/downloads/50m-physical-vectors/50m-rivers-lake-centerlines/](http://www.naturalearthdata.com/downloads/50m-physical-vectors/50m-rivers-lake-centerlines/)), donde cada río es una (multi-)lí


In [ ]:
rivers = gpd.read_file(r"C:\Users\jorge\Downloads\ne_50m_rivers_lake_centerlines.zip")
rivers = rivers[['featurecla', 'name',  'geometry']]
rivers.head()

In [ ]:
rivers.explore()

In [ ]:
print(rivers.geometry[0])

Volvemos a cargar el dataset de countries porque se modifico en los pasos anteriores (pasando a metros).

In [ ]:
countries = gpd.read_file(r"C:\Users\jorge\Downloads\ne_110m_admin_0_countries.zip")
countries = countries[['ADM0_A3', 'SOVEREIGNT', 'CONTINENT', 'POP_EST', 'GDP_MD', 'geometry']]

## Trazando diferentes capas juntas

In [ ]:
# fig, ax = plt.subplots(figsize=(15, 10))
ax = countries.plot(edgecolor='k', facecolor='none', figsize=(15, 10))
rivers.plot(ax=ax)

cities.plot(ax=ax, color='red')
ax.set(xlim=(-140, -20), ylim=(-60, 15))

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd


country = countries[countries['SOVEREIGNT'] == 'Argentina']

# Reproyectar ríos y ciudades al mismo CRS si es necesario
rivers = rivers.to_crs(country.crs) # con .crs consulta el tipo de CRS (coordenadas de referncia) del dataframe
cities = cities.to_crs(country.crs)


fig, ax = plt.subplots(figsize=(15, 10))

country.plot(ax=ax, edgecolor='k', facecolor='none')
rivers.plot(ax=ax, color='blue', linewidth=1)
cities.plot(ax=ax, color='red', markersize=10)

ax.set_xlim(country.total_bounds[0] - 2, country.total_bounds[2] + 2)
ax.set_ylim(country.total_bounds[1] - 2, country.total_bounds[3] + 2)

ax.set_title('Mapa con ríos y ciudades', fontsize=15, fontweight='bold')
plt.show()


## Ejercicios de práctica

A lo largo de los ejercicios de este curso, trabajaremos con varios conjuntos de datos sobre la ciudad de París.

Empezaremos con el siguientes conjuntos de datos:

- Los distritos administrativos de París (https://opendata.paris.fr/explore/dataset/quartier_paris/): `paris_districts_utm.geojson`.


El conjuntos de datos se proporciona como conjunto de datos espaciales en formato de archivo SIG.


**EJERCICIO**:

A continuación, exploraremos el conjunto de datos sobre los distritos administrativos de París (disponible como un archivo GeoJSON: "paris_districts_utm.geojson").

* Lee el conjunto de datos en un GeoDataFrame llamado `districts`.
* Revisa las primeras filas del dataframe. ¿Qué tipo de geometrías contiene este conjunto de datos?
* ¿Cuántas características (features) hay en el conjunto de datos? 
* Seleccione las características `l_qu`, `perimetre`, `st_area_shape`, `geometry`
* Haz un gráfico rápido del conjunto de datos `districts`.


<details><summary>Sugerencias</summary>

* Usa `type(..)` para verificar el tipo de cualquier objeto en Python.
* La función `geopandas.read_file()` puede leer diferentes formatos de archivo geoespacial. Pasa el nombre del archivo como primer argumento.
* Usa el atributo `.shape` para obtener el número de características.

</details>



In [ ]:
districts = gpd.read_file(r"C:\Users\jorge\Downloads\quartier_paris.geojson")
districts.head(1)

In [ ]:
districts.shape

In [ ]:
districts.crs

In [ ]:
districts.plot(figsize=(12, 6))

In [ ]:
districts.explore()

In [ ]:
districts = districts[['l_qu', 'perimetre', 'st_area_shape', 'geometry']]
districts.head(1)

**EJERCICIO**:

¿Cuáles son los distritos más grandes (de mayor área)?

* Calcula el área de cada distrito.
* Añade esta área como una nueva columna al dataframe `districts`.
* Convierta el area representada en metros cuadrados a kilometros cuadrados.
* Ordena el dataframe por esta columna de área de mayor a menor (descendente).
* Cambie la representación usando `to_crs(epsg=3035)`.




**EPSG:3035** es el código EPSG para el European Terrestrial Reference System 1989 / Lambert Azimuthal Equal-Area (ETRS89 / LAEA Europe). Esta es una proyección cartográfica que está diseñada específicamente para el continente europeo y se utiliza comúnmente en análisis geoespaciales en toda Europa.

In [ ]:
districts = districts.to_crs(epsg=3035)
districts.geometry.area

In [ ]:
# dividing by 1^6 for showing km²
#1 kilómetro cuadrado = 1,000,000 metros cuadrados
districts['area'] = districts.geometry.area / 1e6
districts.head()

In [ ]:
# districts.sort_values(by='area', ascending=False)

In [ ]:
districts.plot(column='area', figsize=(12, 6), cmap = 'jet', legend=True)